# Feature Engineering

Building visitor x quarter features from the GA Customer Revenue data. The goal here
is event-volume forecasting (sessions/events per quarter), not revenue - this mirrors
the SMAPI use case, so `transactionRevenue` is intentionally not used.

In [1]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## Load raw data

Only pulling the columns we actually need. `device`, `trafficSource`, and `totals`
arrive as JSON strings, so they get flattened below.

In [2]:
usecols = ["fullVisitorId", "date", "device", "trafficSource", "totals", "geoNetwork"]
df = pd.read_csv(DATA_RAW / "train_v2.csv.zip", usecols=usecols, dtype={"fullVisitorId": str})
df.shape

(1708337, 6)

In [3]:
# Flatten the JSON columns into plain fields, keeping only what we need
device = df["device"].apply(json.loads)
traffic = df["trafficSource"].apply(json.loads)
totals = df["totals"].apply(json.loads)
geo = df["geoNetwork"].apply(json.loads)

df["platform"] = device.apply(lambda d: d.get("operatingSystem"))
df["device_category"] = device.apply(lambda d: d.get("deviceCategory"))
df["traffic_medium"] = traffic.apply(lambda d: d.get("medium"))
df["traffic_source"] = traffic.apply(lambda d: d.get("source"))
# market proxy - geography stands in for "market" since there's no product market field
df["market"] = geo.apply(lambda d: d.get("country"))

# Sparse fields (only present when non-zero) - default to 0
df["sessions"] = totals.apply(lambda d: int(d.get("visits", 0)))
df["events"] = totals.apply(lambda d: int(d.get("hits", 0)))
df["pageviews"] = totals.apply(lambda d: int(d.get("pageviews", 0)))

df = df.drop(columns=["device", "trafficSource", "totals", "geoNetwork"])
df.head()

,date,fullVisitorId,platform,device_category,traffic_medium,traffic_source,market,sessions,events,pageviews
0,20171016,3162355547410993243,Windows,desktop,organic,google,Germany,1,1,1
1,20171016,8934116514970143966,Chrome OS,desktop,referral,sites.google.com,United States,1,2,2
2,20171016,7992466427990357681,Android,mobile,(none),(direct),United States,1,2,2
3,20171016,9075655783635761930,Windows,desktop,organic,google,Turkey,1,2,2
4,20171016,6960673291025684308,Windows,desktop,organic,google,Mexico,1,2,2


Our target variable would be the number of events to predict. Although the logic is different here, hits in Google Analytics can be page visits so a session can have 3 hits (home, products, checkout) and this counts as 3 events. In SMAPI logic, events can be generated from page views, page dom loaded, stay time etc.

In [4]:
# Build the quarter each row belongs to
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
df["quarter"] = df["date"].dt.to_period("Q")

# quarter_of_year (1-4) repeats every year, so it can capture seasonal effects
# (e.g. Dec promotions, summer spikes) that the running `quarter` index can't -
# `quarter` only captures a linear trend over time, not a repeating yearly pattern.
df["quarter_of_year"] = df["date"].dt.quarter

## Aggregate to visitor x quarter

This is the grain the model will train on: one row per visitor per quarter.

In [5]:
features = (
    df.groupby(["fullVisitorId", "quarter"])
    .agg(
        quarter_of_year=("quarter_of_year", "first"),
        total_sessions=("sessions", "sum"),
        total_events=("events", "sum"),
        total_pageviews=("pageviews", "sum"),
        # most common category per visitor-quarter
        platform=("platform", lambda s: s.mode().iat[0] if not s.mode().empty else None),
        device_category=("device_category", lambda s: s.mode().iat[0] if not s.mode().empty else None),
        traffic_medium=("traffic_medium", lambda s: s.mode().iat[0] if not s.mode().empty else None),
        traffic_source=("traffic_source", lambda s: s.mode().iat[0] if not s.mode().empty else None),
        market=("market", lambda s: s.mode().iat[0] if not s.mode().empty else None),
    )
    .reset_index()
)

# average events per session, guarding against divide-by-zero
features["events_per_session"] = features["total_events"] / features["total_sessions"].replace(0, pd.NA)
features.head()

,fullVisitorId,quarter,quarter_of_year,total_sessions,total_events,total_pageviews,platform,device_category,traffic_medium,traffic_source,market,events_per_session
0,0000000259678714014,2017Q4,4,2,19,13,Macintosh,desktop,organic,google,United States,9.5
1,0000010278554503158,2016Q4,4,1,11,8,Macintosh,desktop,organic,google,New Zealand,11.0
2,0000020424342248747,2016Q4,4,1,17,13,Windows,desktop,(none),(direct),Peru,17.0
3,0000027376579751715,2017Q1,1,1,6,5,Macintosh,desktop,(none),(direct),United States,6.0
4,0000039460501403861,2017Q1,1,1,2,2,Windows,desktop,referral,youtube.com,Brazil,2.0


## Lag, quarter-on-quarter change, and target features

Computed per visitor, sorted by quarter, so each row can see its own prior quarter.

In [6]:
features = features.sort_values(["fullVisitorId", "quarter"])

grouped = features.groupby("fullVisitorId")
features["total_events_prev_q"] = grouped["total_events"].shift(1)

# 0 = "no observed history" (e.g. a visitor/SDK's first quarter), a real state rather
# than a missing value - keep this imputation identical in any future scoring pipeline
# to avoid train/serve skew.
features["total_events_prev_q"] = features["total_events_prev_q"].fillna(0)
features["total_events_qoq_change"] = features["total_events"] - features["total_events_prev_q"]

# Target: next quarter's total_events. A visitor's most recent quarter has no future
# quarter yet, so its target is NaN - drop/hold these out at modelling time, don't impute.
features["target_next_q_events"] = grouped["total_events"].shift(-1)
features.head()

,fullVisitorId,quarter,quarter_of_year,total_sessions,total_events,total_pageviews,platform,device_category,traffic_medium,traffic_source,market,events_per_session,total_events_prev_q,total_events_qoq_change,target_next_q_events
0,0000000259678714014,2017Q4,4,2,19,13,Macintosh,desktop,organic,google,United States,9.5,0.0,19.0,NaN
1,0000010278554503158,2016Q4,4,1,11,8,Macintosh,desktop,organic,google,New Zealand,11.0,0.0,11.0,NaN
2,0000020424342248747,2016Q4,4,1,17,13,Windows,desktop,(none),(direct),Peru,17.0,0.0,17.0,NaN
3,0000027376579751715,2017Q1,1,1,6,5,Macintosh,desktop,(none),(direct),United States,6.0,0.0,6.0,NaN
4,0000039460501403861,2017Q1,1,1,2,2,Windows,desktop,referral,youtube.com,Brazil,2.0,0.0,2.0,NaN


## Quarter-level market features

Not per visitor - a single value per quarter, e.g. total active (distinct) visitors.

In [7]:
quarterly_active_users = (
    features.groupby("quarter")["fullVisitorId"].nunique().rename("active_users").reset_index()
)
quarterly_active_users

,quarter,active_users
0,2016Q3,118180
1,2016Q4,241169
2,2017Q1,157501
3,2017Q2,154793
4,2017Q3,192908
5,2017Q4,227582
6,2018Q1,204427
7,2018Q2,66750


## Save

In [8]:
features.to_parquet(DATA_PROCESSED / "visitor_quarter_features.parquet", index=False)
quarterly_active_users.to_parquet(DATA_PROCESSED / "quarterly_active_users.parquet", index=False)